In [1]:
# imports

import os
import openai
import logging
openai.api_key = os.environ["OPENAI_API_KEY"]

In [51]:
# general definitions and helper functions

SPLITTER = '\n'

# generations per question
n_regenerate = 4


def oai_predict(prompt):
    if isinstance(prompt, str):
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": prompt},
        ]
    else:
        messages = prompt
    
    output = openai.ChatCompletion.create(
        model='gpt-3.5-turbo',
        messages=messages,
        max_tokens=200,
    )
    response = output['choices'][0]['message']['content']
    return response


def log_w_indent(text, indent):
    logging.info((indent * 2) * '>>' + ' ' + text)

def predict_w_log(prompt, indent):
    log_w_indent(f'Input: {prompt}', indent)
    response = oai_predict(prompt)
    log_w_indent(f'Output: {response}', indent)
    return response

def setup_logger():
    """Setup logger to always print time and level."""
    logging.basicConfig(
        format='%(asctime)s %(levelname)-8s %(message)s',
        level=logging.INFO,
        datefmt='%Y-%m-%d %H:%M:%S')
    logging.getLogger().setLevel(logging.INFO)  # logging.DEBUG
setup_logger()

def divider(symbol = '*'):
    logging.info(80 * symbol)


base_original_question = '{entity_type} is {entity}?'
base_initial_prompt = "Respond with six short sentences to the following question. Provide concrete facts rather than vague descrptions. Replace periods with '\n'.\n{original_question}"
base_gen_questions_prompt = 'In the context of the question "{original_question}" please list three questions that might have the answer "{fact}".\nYour questions should avoid using specific facts in the answer, but can be specific to the context implied by the original question.'
base_answer_question_prompt = 'In the context of "{original_question}" respond as concisely as possible to the following question: "{question}"'

base_equivalence_prompt = 'The following sentences are answers to the question "{original_question}":'
base_equivalence_prompt += f'\n1. ' + '{fact}'
for i in range(2, n_regenerate + 2):
    base_equivalence_prompt += f'\n{i}. ' + '{}'
base_equivalence_prompt += '\nRespond only with "yes" or "no". In the context of the question "{original_question}", do the above sentences express the same fact? Pay attention to details.'

In [17]:
# Ilya Sutskever
# Yann LeCun
# Pieter Abbeel
# Fei-Fei Li
# Chelsea Finn
# Dawn Song
# Zhang Tong
# Tim Rocktäschel
# Anca Dragan
# George Konidaris

In [18]:
results = dict()
results['prompts'] = dict(
    base_original_input = base_original_input,
    base_initial_prompt = base_initial_prompt,
    base_gen_questions_prompt = base_gen_questions_prompt,
    base_answer_question_prompt = base_answer_question_prompt,
    base_equivalence_prompt = base_equivalence_prompt,
)
logging.info(f'Using prompts {results["prompts"]}')

2023-09-26 14:19:09 INFO     Using prompts {'base_original_input': '{entity_type} is {entity}?', 'base_initial_prompt': "Respond with six short sentences to the following question. Provide concrete facts rather than vague descrptions. Replace periods with '\n'.\n{original_question}", 'base_gen_questions_prompt': 'In the context of the question "{original_question}" please list three questions that might have the answer "{fact}".\nYour questions should avoid using specific facts in the answer, but can be specific to the context implied by the original question.', 'base_answer_question_prompt': 'In the context of "{original_question}" respond as concisely as possible to the following question: "{question}"', 'base_equivalence_prompt': '\n1. {fact}\n2. {}\n3. {}\n4. {}\n5. {}\nRespond with yes or no. Do the above sentences about {entity} express the same fact? Pay attention to details.'}


In [23]:
# replace this with loop over entities
entity = 'Yarin Gal'
# entity = 'Ilya Sutskever'
# entity = 'Chelsea Finn'
entity_type = 'Who'
results[entity] = dict()
e_results = results[entity]
e_results['entity_type'] = entity_type

In [20]:
divider()
log_w_indent(f'Starting with entity {entity}, type {entity_type}', 0)

2023-09-26 14:19:13 INFO     ********************************************************************************
2023-09-26 14:19:13 INFO      Starting with entity Yarin Gal, type Who


In [25]:
original_question = base_initial_question.format(entity_type=entity_type, entity=entity)
intitial_prompt = base_initial_prompt.format(original_question=original_question)
e_results['initial'] = predict_w_log(intitial_prompt, 1)

# set up new results
e_results['final'] = []

2023-09-26 14:20:09 INFO     >>>> Input: Respond with six short sentences to the following question. Provide concrete facts rather than vague descrptions. Replace periods with '
'.
Who is Yarin Gal?
2023-09-26 14:20:14 INFO     >>>> Output: Yarin Gal is a software engineer and entrepreneur. 
He has a background in computer science. 
Yarin Gal is known for his work on probabilistic programming and machine learning. 
He is a lecturer at the University of Oxford. 
Yarin Gal has co-founded a company called OpenAI. 
He specializes in developing algorithms and models for artificial intelligence.


In [57]:
# split response into facts
facts = [(r + SPLITTER) for r in e_results['initial'].split(SPLITTER) if r]
facts = [f.replace('\n', '').strip() for f in facts]

# fill in later
e_results['uncertainty'] = {f'fact-{i}': {} for i in range(len(facts))}

results[entity]['facts'] = facts
log_w_indent(f'Extracted facts: {facts}', 0)

2023-09-26 14:33:52 INFO      Extracted facts: ['Yarin Gal is a software engineer and entrepreneur.', 'He has a background in computer science.', 'Yarin Gal is known for his work on probabilistic programming and machine learning.', 'He is a lecturer at the University of Oxford.', 'Yarin Gal has co-founded a company called OpenAI.', 'He specializes in developing algorithms and models for artificial intelligence.']


In [28]:
# replace this with loop over facts
fidx = 4
fact = facts[fidx]

log_w_indent(f'Currently dealing with fact {fidx}: {fact}', 2)

2023-09-26 14:20:26 INFO     >>>>>>>> Currently dealing with fact 4: Yarin Gal has co-founded a company called OpenAI.


In [31]:
print(base_gen_questions_prompt.format(original_question=original_question, fact=fact))

In the context of the question "Who is Yarin Gal?" please list three questions that might have the answer "Yarin Gal has co-founded a company called OpenAI.".
Your questions should avoid using specific facts in the answer, but can be specific to the context implied by the original question.


In [34]:
gen_questions = predict_w_log(base_gen_questions_prompt.format(original_question=original_question, fact=fact), 3)

e_results['questions'] = {f'fact-{fidx}':  {'question': question, 'answers': []}}

2023-09-26 14:21:33 INFO     >>>>>>>>>>>> Input: In the context of the question "Who is Yarin Gal?" please list three questions that might have the answer "Yarin Gal has co-founded a company called OpenAI.".
Your questions should avoid using specific facts in the answer, but can be specific to the context implied by the original question.
2023-09-26 14:21:36 INFO     >>>>>>>>>>>> Output: 1. What notable achievement or venture is associated with Yarin Gal?
2. What is the name of the company that Yarin Gal has co-founded?
3. Can you provide any information about Yarin Gal's role in a particular organization or project?


In [37]:
questions = [q[3:] for q in gen_questions.split('\n')]
questions = [i for i in questions if i]

log_w_indent(f'Extracted questions: {questions}', 1)

2023-09-26 14:22:11 INFO     >>>> Extracted questions: ['What notable achievement or venture is associated with Yarin Gal?', 'What is the name of the company that Yarin Gal has co-founded?', "Can you provide any information about Yarin Gal's role in a particular organization or project?"]


In [38]:
# replace with loop over questions
qidx = 1

question = questions[qidx]
e_results['questions'][f'fact-{fidx}'][f'question-{qidx}'] = {'question': question, 'answers': []}


answers = e_results['questions'][f'fact-{fidx}'][f'question-{qidx}']['answers']

In [52]:
log_w_indent(f'Regenerate answers for question {qidx} "{question}":', 2)
for re_gen in range(n_regenerate):
    answer = predict_w_log(base_answer_question_prompt.format(original_question=original_question, question=question), 3)
    answers.append(answer)

2023-09-26 14:26:55 INFO     >>>>>>>> Regenerate answers for question 1 "What is the name of the company that Yarin Gal has co-founded?":
2023-09-26 14:26:55 INFO     >>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What is the name of the company that Yarin Gal has co-founded?"
2023-09-26 14:26:56 INFO     >>>>>>>>>>>> Output: OpenAI.
2023-09-26 14:26:56 INFO     >>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What is the name of the company that Yarin Gal has co-founded?"
2023-09-26 14:26:58 INFO     >>>>>>>>>>>> Output: The name of the company that Yarin Gal has co-founded is Vestwell.
2023-09-26 14:26:58 INFO     >>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What is the name of the company that Yarin Gal has co-founded?"
2023-09-26 14:27:00 INFO     >>>>>>>>>>>> Output: T

In [59]:
equiv_prompt = base_equivalence_prompt.format(fact=fact, original_question=original_question, *answers)
equiv_response = predict_w_log(equiv_prompt, 2)

2023-09-26 14:34:32 INFO     >>>>>>>> Input: The following sentences are answers to the question "Who is Yarin Gal?":
1. Yarin Gal has co-founded a company called OpenAI.
2. Yarin Gal has co-founded Outbrain.
3. OpenAI.
4. The name of the company Yarin Gal has co-founded is Lumigo.
5. OpenAI
Respond only with "yes" or "no". In the context of the question "Who is Yarin Gal?", do the above sentences express the same fact? Pay attention to details.
2023-09-26 14:34:33 INFO     >>>>>>>> Output: No.


In [64]:
votes = e_results['uncertainty'][f'fact-{fidx}'][f'question-{qidx}']

binary_response = equiv_response.lower()[:10]
if 'yes' in binary_response:
    e_results['uncertainty'][f'fact-{fidx}'].append(0)
elif 'no' in binary_response:
    e_results['uncertainty'][f'fact-{fidx}'].append(1)
else:
    # how to handle this?
    raise

In [ ]:
# present final results

In [66]:
log_w_indent('Final generation with uncertainty', 1)
for fidx, fact in enumerate(facts):
    uncertainties = ''
    for qidx in e_results['questions'][f'fact-{fidx}']:
        answers = e_results['questions'][f'fact-{fidx}']['answers']
        uncertainties += f'{sum(answers)}/ {len(answers)} '

    log_w_indent(f'(Uncertainty: {uncertainties}) {fact}')

2023-09-26 14:40:31 INFO     >>>> Final generation with uncertainty


KeyError: 'fact-0'